# Week 5 Coding

Two tasks, each connected to the problem set: **C1** ↔ P12; **C2** ↔ P13.

**Predict first.** Before running any check cell, write down on paper what the output will be.

**How to work in this notebook.** Execute the cells in order with Shift+Enter. Each task has a cell with gaps marked `...`: fill them, run the cell, then execute the check cell below it. A passing check prints a confirmation; a failing one raises an `AssertionError`.

## C1: The pole-mode correspondence, executed *(↔ P12)*

The second-order system with the conjugate pole pair $r\,e^{\pm j\theta}$ and the numerator $z^2$ has the recursion $y[n] = 2r\cos\theta\,y[n-1] - r^2\,y[n-2] + u[n]$. Its impulse response is computed by iterating the recursion with the input $\delta[n]$. The design follows the procedure of P12, applied to a different specification: a period of $8$ samples gives $\theta = 2\pi/8 = \pi/4$, and an envelope reduced by a factor $100$ after $20$ samples gives $r = 0.01^{1/20} = 0.794$.

a) Complete the recursion in `ring`.

b) The check cell runs the design $r = 0.794$, $\theta = \pi/4$ (period $8$ samples, envelope reduced by a factor $100$ after $20$ samples) and two variants of it, and verifies the map's predictions: the envelope of the design falls to about $1\%$ of the peak at $n = 20$, the sign pattern repeats every $8$ samples, the pole pair on the unit circle never decays, and the pair outside it grows.

c) The plot cell draws the five impulse responses; compare the period and the envelope of each with its pole pair.

In [ ]:
import numpy as np

def ring(r, theta, N=60):
    """Impulse response of the causal system with poles r e^{+-j theta} and numerator z^2."""
    a1, a2 = 2*r*np.cos(theta), r**2
    h = np.zeros(N); h[0] = 1.0
    for n in range(1, N):
        h[n] = a1*h[n-1] - (a2*h[n-2] if n >= 2 else 0.0)
    return h

cases = [(0.794, np.pi/4, "design: period 8, 1% in 20"), (0.631, np.pi/4, "variant: 1% in 10"),
         (0.794, np.pi/2, "variant: period 4"), (1.00, np.pi/4, "on the unit circle"), (1.02, np.pi/4, "outside")]
for r, th, label in cases:
    h = ring(r, th)
    print(f"{label:24s} peak = {np.max(np.abs(h)):6.2f}   |h[20]|/peak = {abs(h[20])/np.max(np.abs(h)):.4f}   |h[40]| = {abs(h[40]):.4f}")

In [ ]:
h = ring(0.794, np.pi/4)
assert np.allclose(h[:3], [1.0, 1.123, 0.631], atol=2e-3)                 # the design chain: h[0], h[1], h[2]
assert abs(h[20]) / np.max(np.abs(h)) < 0.012                            # envelope at 1% of the peak after 20 samples
assert np.allclose(h[[3, 7, 11]], 0) and h[1] > 0 > h[5] and h[9] > 0   # zero crossings every 4 samples: period 8
assert abs(ring(0.631, np.pi/4)[10]) / 1.0 < 0.012                       # the variant r = 0.631: 1% after 10 samples
hb = ring(0.794, np.pi/2); assert np.allclose(hb[1::2], 0)               # the variant theta = pi/2: every other sample is zero (period 4)
hc = ring(1.00, np.pi/4); assert abs(np.max(np.abs(hc[40:])) - np.max(np.abs(hc[:20]))) < 0.05   # on the circle: no decay
assert np.max(np.abs(ring(1.02, np.pi/4)[40:])) > np.max(np.abs(ring(1.02, np.pi/4)[:8]))         # outside: growth
print("C1 passed: the map's predictions hold; the radius sets the envelope, the angle the period.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(14, 2.4))
for ax, (r, th, label) in zip(axes, cases):
    h = ring(r, th, 40)
    ax.stem(np.arange(40), h)
    ax.set_title(f"$r = {r}$, $\\theta = {th/np.pi:.2f}\\pi$", fontsize=9); ax.set_xlabel("$n$")
plt.tight_layout()

## C2: The magnitude of $H(z)$ over the plane *(↔ P13)*

The transfer function of the design of C1, $H(z) = z^2 / \bigl((z - p_1)(z - p_2)\bigr)$ with $p_{1,2} = 0.794\,e^{\pm j\pi/4}$, is evaluated on a grid of the complex plane and its magnitude is drawn as a contour map, together with the unit circle.

a) Complete the expression for $H$ on the grid.

b) The check cell evaluates $\lvert H\rvert$ along the unit circle and verifies that its maximum lies at the angle of the pole pair, within $0.05$ radians, and that moving the poles outward to radius $0.977$ raises that maximum by about a factor of eight. Week 7 studies exactly this evaluation along the circle.

c) The plot cell draws the contour map; the unit circle passes closest to the poles at the angle $\pm\pi/4$.

In [ ]:
re, im = np.meshgrid(np.linspace(-1.5, 1.5, 400), np.linspace(-1.5, 1.5, 400))
z = re + 1j*im

def H_of(z, r, theta):
    """H(z) = z^2 / ((z - p1)(z - p2)) with the conjugate pair p1, p2 = r e^{+-j theta}."""
    p1, p2 = r*np.exp(1j*theta), r*np.exp(-1j*theta)
    return z**2 / ((z - p1) * (z - p2))

r, theta = 0.794, np.pi/4
H = H_of(z, r, theta)
print("|H| at z = 1 (DC):", abs(H_of(1.0, r, theta)).round(3), "  at z = -1:", abs(H_of(-1.0, r, theta)).round(3))

In [ ]:
W = np.linspace(0, np.pi, 2001)
gain = np.abs(H_of(np.exp(1j*W), r, theta))                 # |H| along the unit circle
assert abs(W[np.argmax(gain)] - np.pi/4) < 0.05             # the maximum sits at the angle of the pole pair
gain_near = np.abs(H_of(np.exp(1j*W), 0.977, theta))
assert 7 < gain_near.max() / gain.max() < 9                 # poles moved outward: the maximum grows by about 8
assert abs(abs(H_of(1.0, r, theta)) - 1.97) < 0.02          # the DC gain 1.97 of this design, used again in Week 7
print(f"C2 passed: max |H| = {gain.max():.2f} at angle {W[np.argmax(gain)]:.3f} rad; at r = 0.977 it is {gain_near.max():.1f}.")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.6))
cf = ax.contourf(re, im, np.minimum(np.abs(H), 10), levels=40)
circle = np.exp(1j*np.linspace(0, 2*np.pi, 300))
ax.plot(circle.real, circle.imag, "w--", lw=2)               # the unit circle
ax.plot([r*np.cos(theta), r*np.cos(theta)], [r*np.sin(theta), -r*np.sin(theta)], "wx", ms=8)
ax.set_aspect("equal"); ax.set_xlabel("Re"); ax.set_ylabel("Im")
fig.colorbar(cf, label="$|H(z)|$, clipped at 10")
ax.set_title("$|H(z)|$ over the plane; poles at $0.794\\,e^{\\pm j\\pi/4}$", fontsize=10)
plt.tight_layout()